# 43 - Exploring and lightly cleaning the random 300K sample from Kerem

Kerem's file arrived as `dataset/part-0000.parquet` (a generic Spark/Hadoop-style output name), renamed here to `dataset/goi_random_sample_300k.parquet` so it's clear what it is. This is the fully random, query-independent sample requested for two later uses: (1) a scalability check, embedding the existing corpus and gold standard in a much larger, mostly-irrelevant pool to see how retrieval quality and latency hold up, and (2) a coverage-gap check, seeing whether any of these companies score highly against existing queries despite never having been surfaced by production.

This notebook does neither of those yet, it profiles the raw data (schema, missing values, distributions, overlap with the existing corpus) and produces a lightly cleaned copy, so the encoding stage that comes next has a known-good starting point. Requires `pyarrow` (`pip install pyarrow`) to read the Parquet file.

In [2]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("dataset/goi_random_sample_300k.parquet")
OUTPUT_DIR = Path("result/43_random_sample_exploration")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns from {DATA_PATH}")
print()
print("Schema:")
print(df.dtypes)

Loaded 300000 rows, 11 columns from dataset/goi_random_sample_300k.parquet

Schema:
domain               object
name                 object
organization_type    object
organization_size    object
country              object
state                object
district             object
municipality         object
summary              object
summary_keywords     object
nace_code            object
dtype: object


In [3]:
# Missing values -- state/district/municipality sparsity is expected (many countries don't
# record city-level detail consistently), not a data quality problem on its own.
print("Missing values per column:")
print(df.isna().sum())
print()
print("Missing rate (%):")
print((100 * df.isna().mean()).round(1))

Missing values per column:
domain                    0
name                      0
organization_type         0
organization_size        18
country                  13
state                 47002
district             181068
municipality         226169
summary                   0
summary_keywords          0
nace_code                18
dtype: int64

Missing rate (%):
domain                0.0
name                  0.0
organization_type     0.0
organization_size     0.0
country               0.0
state                15.7
district             60.4
municipality         75.4
summary               0.0
summary_keywords      0.0
nace_code             0.0
dtype: float64


In [4]:
# Categorical distributions -- sanity check that the sample looks like a plausible global
# company population, not skewed toward one type/size/geography by accident.
print("organization_type:")
print(df["organization_type"].value_counts(dropna=False))
print()
print("organization_size:")
print(df["organization_size"].value_counts(dropna=False))
print()
print("Top 15 countries:")
print(df["country"].value_counts(dropna=False).head(15))
print()
print("nace_code (top-level categories):", df["nace_code"].nunique(), "unique values")
print(df["nace_code"].value_counts(dropna=False))

organization_type:
organization_type
Company     246258
Other        45447
Public        4018
Academic      3549
Startup        728
Name: count, dtype: int64

organization_size:
organization_size
Small (10-49)              132366
Micro (0-9)                128597
Medium-sized (50-249)       26496
Large enterprise (250+)     12523
None                           18
Name: count, dtype: int64

Top 15 countries:
country
United States     74623
Germany           31841
United Kingdom    19741
China             13424
Netherlands       13188
Australia         11927
Italy             10978
Japan             10517
France            10077
Brazil             9044
Canada             6701
Spain              6670
Russia             6582
Poland             6031
Czechia            4806
Name: count, dtype: int64

nace_code (top-level categories): 21 unique values
nace_code
NACE N: Professional, scientific and technical activities                                                                            

In [5]:
# Domain uniqueness and overlap with the existing 98,716-company corpus. Overlap should be
# small -- this is meant to be an independent, mostly-new sample, not a re-draw of what's
# already there. A large overlap would mean it isn't actually adding new coverage.
print("Duplicate domains:", df["domain"].duplicated().sum())

corpus = pd.read_csv("dataset/company_corpus.csv")
overlap = df["domain"].isin(corpus["domain"])
print(f"Overlap with existing corpus: {overlap.sum()} / {len(df)} ({100 * overlap.mean():.2f}%)")
print(f"New, previously-unseen companies: {(~overlap).sum()} ({100 * (~overlap).mean():.2f}%)")

Duplicate domains: 0
Overlap with existing corpus: 1691 / 300000 (0.56%)
New, previously-unseen companies: 298309 (99.44%)


In [6]:
# Summary text quality -- length distribution and exact-duplicate summaries (a soft signal
# of templated/boilerplate company pages, same concern trust_feature.py checks for elsewhere
# in this thesis, though not the same detector).
print("Summary length (characters) stats:")
print(df["summary"].str.len().describe())
print()
n_dup_summary = df["summary"].duplicated(keep=False).sum()
print(f"Rows sharing an exact-duplicate summary with at least one other row: {n_dup_summary} ({100 * n_dup_summary / len(df):.2f}%)")
print("Not dropped -- flagged only, since a shared summary doesn't necessarily mean an invalid row, some companies do have near-identical boilerplate pages.")

Summary length (characters) stats:
count    300000.000000
mean        418.988223
std          63.412634
min         107.000000
25%         378.000000
50%         418.000000
75%         459.000000
max         938.000000
Name: summary, dtype: float64

Rows sharing an exact-duplicate summary with at least one other row: 3094 (1.03%)
Not dropped -- flagged only, since a shared summary doesn't necessarily mean an invalid row, some companies do have near-identical boilerplate pages.


## Light cleaning

Nothing here needed heavy fixing: no duplicate domains, no literal-string "None" placeholders (missing values are genuine nulls), no empty summaries. The only things worth doing before encoding: flag (not drop) duplicate-summary rows and rows already present in the existing corpus, so downstream steps can filter them in or out deliberately rather than by accident, and save a cleaned copy under a clear name. The raw file (`goi_random_sample_300k.parquet`) is left untouched.

In [7]:
clean_df = df.copy()
clean_df["already_in_existing_corpus"] = clean_df["domain"].isin(corpus["domain"])
clean_df["has_duplicate_summary"] = clean_df["summary"].duplicated(keep=False)

clean_path = Path("dataset/goi_random_sample_300k_clean.parquet")
clean_df.to_parquet(clean_path, index=False)

print(f"Saved cleaned copy -> {clean_path}")
print(f"Rows: {len(clean_df)}")
print(f"Flagged already_in_existing_corpus: {clean_df['already_in_existing_corpus'].sum()}")
print(f"Flagged has_duplicate_summary: {clean_df['has_duplicate_summary'].sum()}")

Saved cleaned copy -> dataset/goi_random_sample_300k_clean.parquet
Rows: 300000
Flagged already_in_existing_corpus: 1691
Flagged has_duplicate_summary: 3094
